# Explore San Antonio 311 data

This notebook connects the familiar Jupyter workflow to the reusable code in `ingest.py`. Run the cells from top to bottom. The notebook downloads a small current snapshot, loads it into pandas, and asks basic data-quality questions before any transformation logic is written.

## 1. Import the tools

Imports make code from libraries and our `ingest.py` module available in this notebook. This is the notebook equivalent of reusing functions rather than copying their code.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from san_antonio_311.explore import (
    category_counts,
    convert_date_columns,
    duplicate_request_report,
    missing_values_report,
    summarize_dataset,
)
from san_antonio_311.ingest import fetch_features, write_jsonl

## 2. Download a small sample

The command-line program calls these same two functions. Keeping the sample small makes exploration quick and avoids requesting more data than we need while learning.

In [ ]:
sample_size = 100
output_path = Path("../data/raw/notebook_sample.jsonl")

records = fetch_features(limit=sample_size)
write_jsonl(records, output_path)

print(f"Downloaded {len(records)} records to {output_path}")

## 3. Load JSONL into pandas

A pandas `DataFrame` is a table similar to a spreadsheet. `lines=True` tells pandas that each line in the file is one JSON record.

In [ ]:
df = pd.read_json(output_path, lines=True)
df.head()

## 4. Understand the table's shape

The reusable `summarize_dataset()` helper reports the table dimensions, columns, and pandas data types.

In [ ]:
summary = summarize_dataset(df)
summary

## 5. Inspect data types

Data types determine which operations are safe. Dates may require explicit conversion later because the raw layer represents them as epoch milliseconds.

In [ ]:
pd.Series(summary["data_types"], name="pandas_type").to_frame()

## 6. Look for missing values

Missing values are not automatically errors, but we need to understand where they occur before defining validation rules.

In [ ]:
missing = missing_values_report(df)
missing[missing["missing_count"] > 0]

## 7. Check request numbers and convert dates

Duplicate request numbers deserve investigation. Date conversion creates a new DataFrame, preserving the raw table for comparison.

In [ ]:
duplicates = duplicate_request_report(df)
dated_df = convert_date_columns(df)

print(f"Duplicate rows: {len(duplicates)}")
dated_df[["CREATE_DATE", "DUE_DATE", "CLOSED_DATE", "LAST_UPDATED"]].head()

## 8. Count requests by status and category

Counts help us check whether categorical fields contain the kinds of values we expect.

In [ ]:
status_counts = df["STATUS"].value_counts(dropna=False)
category_summary = category_counts(df, top_n=10)

display(status_counts.to_frame(name="requests"))
display(category_summary.to_frame())

## 9. Visualize the most common categories

A chart can reveal patterns that are less obvious in a table. Remember that this small snapshot is not enough to make conclusions about long-term city trends.

In [ ]:
top_categories = category_summary.sort_values()

ax = top_categories.plot(kind="barh", figsize=(9, 5), color="#8b0e04")
ax.set_title("Most common 311 categories in this sample")
ax.set_xlabel("Number of requests")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

## What we learned

This notebook is for investigation. Once we decide on repeatable rules—such as converting dates, enforcing required identifiers, or standardizing categories—we will move those rules into a tested `transform.py` module. That keeps exploration flexible while making the production pipeline reliable.